# UAE IA Compliance Mapping Pipeline — Google Colab

**Runtime:** GPU → T4 (free tier) ~25–40 min | CPU → ~3–4 hrs  
**Steps:**
1. Clone repo from GitHub
2. Mount Google Drive and copy your data files in
3. Install dependencies
4. Run the pipeline
5. Download results back to Drive

**Change runtime to GPU first:**  
`Runtime → Change runtime type → T4 GPU`

In [3]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('Running on CPU — will be slower (~3-4 hrs). Consider enabling GPU.')

GPU available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Step 2 — Clone the repo

In [5]:
import os

REPO_URL = 'https://github.com/puneetha08nr/regulatory_parsing_improved.git'
REPO_DIR = 'regulatory_parsing_improved'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest changes')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

Cloning into 'regulatory_parsing_improved'...
fatal: could not read Username for 'https://github.com': No such device or address


FileNotFoundError: [Errno 2] No such file or directory: 'regulatory_parsing_improved'

## Step 3 — Mount Google Drive and copy data files

**Before running this cell**, upload these files to your Google Drive under a folder called `compliance_pipeline_data/`:

| File | Where it lives locally |
|------|------------------------|
| `policies/` folder (all `*_for_mapping.json`) | `data/02_processed/policies/` |
| `uae_ia_controls_structured.json` | `data/02_processed/` |
| `golden_mapping_dataset.json` | `data/07_golden_mapping/` |
| `not_applicable_passages.json` | `data/07_golden_mapping/` |
| `obligation-classifier-legalbert/` folder | repo root |

You can zip the `data/` folder and unzip it here, or upload files individually.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/compliance_pipeline_data'

# ── Copy data files from Drive into the repo ──────────────────────────────
import shutil, pathlib
# Unzip model if uploaded as zip
import subprocess
zip_path = f'{DRIVE_DATA}/obligation-classifier-legalbert.zip'
model_dst = f'{REPO_DIR}/obligation-classifier-legalbert'
if pathlib.Path(zip_path).exists() and not pathlib.Path(model_dst).exists():
    subprocess.run(['unzip', '-q', zip_path, '-d', REPO_DIR], check=True)
    print('  ✅ Unzipped obligation-classifier-legalbert')

def copy_from_drive(src, dst):
    src, dst = pathlib.Path(src), pathlib.Path(dst)
    if not src.exists():
        print(f'  SKIP (not found in Drive): {src}')
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print(f'  ✅ {src.name} → {dst}')

# Policy passages
copy_from_drive(f'{DRIVE_DATA}/policies',
                f'{REPO_DIR}/data/02_processed/policies')

# UAE IA controls
copy_from_drive(f'{DRIVE_DATA}/uae_ia_controls_structured.json',
                f'{REPO_DIR}/data/02_processed/uae_ia_controls_structured.json')

# Golden dataset (blocklist + not-applicable passages)
copy_from_drive(f'{DRIVE_DATA}/golden_mapping_dataset.json',
                f'{REPO_DIR}/data/07_golden_mapping/golden_mapping_dataset.json')
copy_from_drive(f'{DRIVE_DATA}/not_applicable_passages.json',
                f'{REPO_DIR}/data/07_golden_mapping/not_applicable_passages.json')

# LegalBERT fine-tuned model (optional — skip if you want rule-based classifier)
copy_from_drive(f'{DRIVE_DATA}/obligation-classifier-legalbert',
                f'{REPO_DIR}/obligation-classifier-legalbert')

print('\nDone copying.')

## Step 4 — Install dependencies

In [ ]:
%%capture install_output
# Core pipeline dependencies only (skip docling/flask/label-studio which are not needed for the pipeline run)
!pip install -q \
    transformers \
    sentence-transformers \
    rank-bm25 \
    pandas \
    scikit-learn \
    numpy \
    sentencepiece

print('Install complete.')
# Show last few lines of install output if there were errors
lines = install_output.stdout.split('\n')
errors = [l for l in lines if 'error' in l.lower() or 'failed' in l.lower()]
if errors:
    print('Potential issues:')
    for e in errors[:5]: print(' ', e)

## Step 5 — Run the pipeline

In [ ]:
import os
os.chdir(REPO_DIR)

# Force CPU_MODE=0 so it uses the full accurate models (we have a GPU now)
os.environ['CPU_MODE'] = '0'
# Use the best reranker on GPU
os.environ['RERANKER_MODEL'] = 'BAAI/bge-reranker-base'
# top_k_per_control=8 for richer output on GPU
os.environ['TOP_K'] = '8'

# Run the pipeline — output files go to data/06_compliance_mappings/
!python3 quick_start_compliance.py

## Step 6 — Evaluate results (Precision / Recall@K / RePASs)

In [ ]:
os.chdir(REPO_DIR)
!python3 scripts/evaluate_pipeline.py

## Step 7 — Save results back to Google Drive

In [ ]:
import shutil, pathlib, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
RESULTS_DIR = f'/content/drive/MyDrive/compliance_pipeline_data/results_{timestamp}'
pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

FILES_TO_SAVE = [
    ('data/06_compliance_mappings/mappings.csv',             'mappings.csv'),
    ('data/06_compliance_mappings/mappings.json',            'mappings.json'),
    ('data/06_compliance_mappings/compliance_report.json',   'compliance_report.json'),
    ('data/06_compliance_mappings/evaluation_report.json',   'evaluation_report.json'),
    ('data/06_compliance_mappings/retrieval_log.json',       'retrieval_log.json'),
]

for src, dst_name in FILES_TO_SAVE:
    src_path = pathlib.Path(REPO_DIR) / src
    if src_path.exists():
        shutil.copy2(src_path, f'{RESULTS_DIR}/{dst_name}')
        print(f'  ✅ saved: {dst_name}')
    else:
        print(f'  ⚠️  not found: {src}')

# Save per-policy folder
by_policy_src = pathlib.Path(REPO_DIR) / 'data/06_compliance_mappings/by_policy'
if by_policy_src.exists():
    shutil.copytree(by_policy_src, f'{RESULTS_DIR}/by_policy')
    print(f'  ✅ saved: by_policy/')

print(f'\nAll results saved to Google Drive: {RESULTS_DIR}')

## (Optional) Step 8 — Generate next annotation batch

Creates a fresh `golden_set_mapping_tasks.json` from the new pipeline output.  
Import this into Label Studio for the next annotation round.

In [ ]:
os.chdir(REPO_DIR)
!python3 create_golden_set_tasks.py \
    --candidates data/06_compliance_mappings/mappings.json \
    --output-tasks data/03_label_studio_input/golden_set_mapping_tasks.json

# Save it to Drive too
shutil.copy2(
    f'{REPO_DIR}/data/03_label_studio_input/golden_set_mapping_tasks.json',
    f'{RESULTS_DIR}/golden_set_mapping_tasks.json'
)
print('Saved: golden_set_mapping_tasks.json → import this into Label Studio')